# Invalid exemplar montages (per category)

**Purpose:** For each category in the same CCN analysis pool as Plot A/B (`data/included_categories.txt` ∩ per-class precision > threshold in `annotation/per_class_validation_data.csv`), list exemplars that **fail** per-file rater precision in `annotation/per_file_precision_data.csv`, and save a montage of crop images for each category that has at least one such file.

**Invalid rule:** `per_file_precision_used <= PRECISION_THRESHOLD` (non-inclusive above threshold), matching the strict filter in `04_exemplar_inclusion_summary_stats.ipynb`. Precision column: `actual_precision` when present, otherwise `precision`.

**Images:** `cropped_dir / filename` where `filename` is the relative path in the per-file CSV (e.g. `cup/cup_0.42_....jpg`). Missing files are skipped.

**Ordering within a montage:** sort by YOLO confidence parsed from the filename stem (high → low), same convention as other notebooks; cap at `max_tiles` (default 25, 5×5 grid with padding).

## Parameters

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SCRIPT_DIR = Path('.').resolve()
PROJECT_ROOT = SCRIPT_DIR.parent.parent

PRECISION_THRESHOLD = 0.6

INCLUDED_CATEGORIES_TXT = PROJECT_ROOT / 'data/included_categories.txt'
PER_CLASS_VALIDATION_CSV = PROJECT_ROOT / 'annotation/per_class_validation_data.csv'
PER_FILE_PRECISION_CSV = PROJECT_ROOT / 'annotation/per_file_precision_data.csv'

cropped_dir = Path(os.getenv('BV_CROPS_BASE', 'SET_BV_CROPS_BASE')).expanduser()
out_dir = SCRIPT_DIR / 'plotE_invalid_exemplar_montages_per_category'

n_cols = 5
max_tiles = n_cols * n_cols
cell_size = (128, 128)

for p in [INCLUDED_CATEGORIES_TXT, PER_CLASS_VALIDATION_CSV, PER_FILE_PRECISION_CSV]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}')

out_dir.mkdir(parents=True, exist_ok=True)
print('Threshold (invalid if file precision <= this):', PRECISION_THRESHOLD)
print('Output dir:', out_dir)

## Helpers

In [ ]:
def load_allowed_categories(included_txt, per_class_csv, threshold=0.6):
    included = set(line.strip().lower() for line in included_txt.read_text().splitlines() if line.strip())
    per_class = pd.read_csv(per_class_csv, usecols=['class', 'precision']).dropna(subset=['class', 'precision']).copy()
    per_class['class'] = per_class['class'].astype(str).str.strip().str.lower()
    per_class['precision'] = pd.to_numeric(per_class['precision'], errors='coerce')
    precision_allowed = set(per_class.loc[per_class['precision'] > threshold, 'class'])
    return included & precision_allowed


def load_per_file_precision(per_file_csv):
    df = pd.read_csv(per_file_csv).dropna(subset=['filename', 'class']).copy()
    if 'actual_precision' in df.columns:
        df['per_file_precision_used'] = pd.to_numeric(df['actual_precision'], errors='coerce')
    elif 'precision' in df.columns:
        df['per_file_precision_used'] = pd.to_numeric(df['precision'], errors='coerce')
    else:
        raise ValueError('Expected either `actual_precision` or `precision` column in per-file CSV.')
    df['class'] = df['class'].astype(str).str.strip().str.lower()
    df['raters'] = pd.to_numeric(df['raters'], errors='coerce')
    df['stem'] = df['filename'].map(lambda s: Path(str(s).strip()).stem)
    return df


def parse_confidence_from_stem(stem):
    parts = str(stem).split('_')
    if len(parts) < 2:
        return np.nan
    try:
        return float(parts[1])
    except ValueError:
        return np.nan


def make_montage(images, n_cols, cell_size):
    if not images:
        return None
    n = len(images)
    n_rows = (n + n_cols - 1) // n_cols
    total_h = n_rows * cell_size[1]
    total_w = n_cols * cell_size[0]
    out = Image.new('RGB', (total_w, total_h), (255, 255, 255))
    for idx, img in enumerate(images):
        row, col = idx // n_cols, idx % n_cols
        if img.size != (cell_size[0], cell_size[1]):
            img = img.resize((cell_size[0], cell_size[1]), Image.Resampling.LANCZOS)
        out.paste(img, (col * cell_size[0], row * cell_size[1]))
    return out

## Select invalid rows (per-file CSV) in analysis categories

In [ ]:
allowed_categories = load_allowed_categories(
    INCLUDED_CATEGORIES_TXT,
    PER_CLASS_VALIDATION_CSV,
    threshold=PRECISION_THRESHOLD,
)
per_file_df = load_per_file_precision(PER_FILE_PRECISION_CSV)
pool = per_file_df[per_file_df['class'].isin(allowed_categories)].copy()

invalid = pool[~(pool['per_file_precision_used'] > PRECISION_THRESHOLD)].copy()
invalid['confidence'] = invalid['stem'].map(parse_confidence_from_stem)

summary = (
    invalid.groupby('class', sort=True)
    .agg(
        n_invalid_files=('filename', 'count'),
        mean_file_precision=('per_file_precision_used', 'mean'),
    )
    .reset_index()
    .rename(columns={'class': 'category'})
    .sort_values('n_invalid_files', ascending=False)
)
summary_path = out_dir / 'invalid_exemplar_counts_by_category.csv'
summary.to_csv(summary_path, index=False)

print(f'Allowed categories: {len(allowed_categories)}')
print(f'Per-file rows in pool: {len(pool):,}')
print(f'Invalid rows (file precision <= {PRECISION_THRESHOLD} or NaN): {len(invalid):,}')
print(f'Categories with >=1 invalid file: {(summary["n_invalid_files"] > 0).sum()}')
print(f'Wrote: {summary_path}')
summary.head(20)

## Build and save montages (one PNG + PDF per category)

In [ ]:
cropped_dir = Path(cropped_dir)
records = []

cats_with_invalid = sorted(invalid['class'].unique())
for cat in tqdm(cats_with_invalid, desc='Categories'):
    g = invalid[invalid['class'] == cat].copy()
    g = g.sort_values('confidence', ascending=False, na_position='last')
    images = []
    used_stems = []
    missing = 0
    for _, row in g.iterrows():
        if len(images) >= max_tiles:
            break
        rel = str(row['filename']).strip()
        path = cropped_dir / rel
        if not path.is_file():
            missing += 1
            continue
        try:
            img = Image.open(path).convert('RGB')
            images.append(img)
            used_stems.append(row['stem'])
        except Exception:
            missing += 1
            continue

    n_loaded = len(images)
    if n_loaded == 0:
        continue

    target_n = max_tiles
    images = images[:target_n]
    while len(images) < target_n:
        images.append(Image.new('RGB', cell_size, (245, 245, 245)))

    montage = make_montage(images, n_cols, cell_size)
    if montage is None:
        continue

    safe_cat = cat.replace(' ', '_')
    base = f'invalid_exemplars_{safe_cat}_nshown{n_loaded}_of_{len(g)}'
    png_path = out_dir / f'{base}.png'
    pdf_path = out_dir / f'{base}.pdf'
    montage.save(png_path)
    montage.save(pdf_path, 'PDF', resolution=300.0)

    records.append({
        'category': cat,
        'n_invalid_in_csv': int(len(g)),
        'n_images_loaded': int(n_loaded),
        'n_missing_or_unreadable': int(missing),
        'montage_png': png_path.name,
        'montage_pdf': pdf_path.name,
        'stems_in_montage_order': ' | '.join(used_stems),
    })

meta = pd.DataFrame(records).sort_values('category')
meta_path = out_dir / 'invalid_exemplar_montage_manifest.csv'
meta.to_csv(meta_path, index=False)
print(f'Wrote {len(meta)} montage pairs under {out_dir}')
print(f'Manifest: {meta_path}')